**9.3 Handling Missing Values Safely (After Train Test Split)**

In this lesson:
- Load the split Titanic data from 9.2
- Explore missing values
- Demonstrate why dropna (dropping rows) is not a good idea (always)
- how to detect columns with high missing percentage
- Fill missing values without data leakage:
    * Median for numeric columns using TRAIN only
    * Mode for categorical columns using TRAIN only
    * Apply the same values to both TRAIN and TEST
- Save the imputed train and test sets

**1. Setup**

In [1]:
# %pip install -q numpy pandas

In [2]:
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)

**2. Load Split Data (from 9.2. train test split)**

In [3]:
train_df = pd.read_csv(r"C:\Users\AYO_AYO\Desktop\Machine-Learning-Bootcamp\Data_Preprocessing\train_df.csv")
test_df = pd.read_csv(r"C:\Users\AYO_AYO\Desktop\Machine-Learning-Bootcamp\Data_Preprocessing\test_df.csv")

print("Train Shape:", train_df.shape)
print("Test Shape:", test_df.shape)

Train Shape: (712, 11)
Test Shape: (179, 11)


In [4]:
print(train_df.shape)
print(train_df.columns.to_list())
train_df.head()

(712, 11)
['pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'adult_male', 'deck', 'embark_town', 'alone', 'survived']


,pclass,sex,age,sibsp,parch,fare,adult_male,deck,embark_town,alone,survived
0,3,male,NaN,0,0,56.4958,True,NaN,Southampton,True,1
1,2,male,NaN,0,0,0.0000,True,NaN,Southampton,True,0
2,1,male,NaN,0,0,221.7792,True,C,Southampton,True,0
3,3,female,18.0,0,1,9.3500,False,NaN,Southampton,False,1
4,2,female,31.0,1,1,26.2500,False,NaN,Southampton,False,1


**3. Separate X and y**

In [5]:
target = "survived"

In [6]:
# training set - features and target variable
X_train = train_df.drop(columns=[target])
y_train = train_df[target]

# test set - features and target variable
X_test = test_df.drop(columns=[target])
y_test = test_df[target]

In [7]:
X_test.head()

,pclass,sex,age,sibsp,parch,fare,adult_male,deck,embark_town,alone
0,3,male,24.0,2,0,24.1500,True,NaN,Southampton,False
1,3,male,44.0,0,1,16.1000,True,NaN,Southampton,False
2,3,male,22.0,0,0,7.2250,True,NaN,Cherbourg,True
3,3,male,41.0,2,0,14.1083,True,NaN,Southampton,False
4,3,female,NaN,1,0,15.5000,False,NaN,Queenstown,False


In [8]:
y_test.head()

0    0
1    0
2    1
3    0
4    1
Name: survived, dtype: int64

**4. Checking missing values before processing**

In [9]:
print("Missing values in TRAIN (before)")
print(X_train.isna().sum())

print("-"*50)

print("Missing values in TEST (before)")
print(X_test.isna().sum())

Missing values in TRAIN (before)
pclass           0
sex              0
age            137
sibsp            0
parch            0
fare             0
adult_male       0
deck           553
embark_town      2
alone            0
dtype: int64
--------------------------------------------------
Missing values in TEST (before)
pclass           0
sex              0
age             40
sibsp            0
parch            0
fare             0
adult_male       0
deck           135
embark_town      0
alone            0
dtype: int64


In [10]:
print("Missing values percentage in TRAIN (before)")
print(X_train.isna().mean() * 100)
print("-"*50)

print("Missing values percentage in TEST (before)")
print(X_test.isna().mean() * 100)

Missing values percentage in TRAIN (before)
pclass          0.000000
sex             0.000000
age            19.241573
sibsp           0.000000
parch           0.000000
fare            0.000000
adult_male      0.000000
deck           77.668539
embark_town     0.280899
alone           0.000000
dtype: float64
--------------------------------------------------
Missing values percentage in TEST (before)
pclass          0.000000
sex             0.000000
age            22.346369
sibsp           0.000000
parch           0.000000
fare            0.000000
adult_male      0.000000
deck           75.418994
embark_town     0.000000
alone           0.000000
dtype: float64


**5. Demonstration: Why dropna (drop the rows) is not usually done in ML pipelines**

In [11]:
print("Demonstration of dropna:")
print("Original TRAIN shape:", X_train.shape)

# make sure to drop the corresponding rows in y_train \ similarity for test data
dropped_demo = X_train.dropna()
print("Shape after dropna:", dropped_demo.shape)

Demonstration of dropna:
Original TRAIN shape: (712, 10)
Shape after dropna: (141, 10)


Observation:

dropna removed all rows that had even one missing value.
This can throw away a lot of training examples.
It also introduces bias because the missingness pattern is not random.
Therefore, dropna is shown only for demonstration and not used in real pipelines.

**6. Demonstration: COlumns with high missing percentage**

In [12]:
print("Missing values percentage in TRAIN (before)")
print(X_train.isna().mean() * 100)

print("-"*50)

print("Missing values percentage in TEST (before)")
print(X_test.isna().mean() * 100)

Missing values percentage in TRAIN (before)
pclass          0.000000
sex             0.000000
age            19.241573
sibsp           0.000000
parch           0.000000
fare            0.000000
adult_male      0.000000
deck           77.668539
embark_town     0.280899
alone           0.000000
dtype: float64
--------------------------------------------------
Missing values percentage in TEST (before)
pclass          0.000000
sex             0.000000
age            22.346369
sibsp           0.000000
parch           0.000000
fare            0.000000
adult_male      0.000000
deck           75.418994
embark_town     0.000000
alone           0.000000
dtype: float64


In [13]:
missing_threshold = 40

missing_percent_train = X_train.isna().mean() * 100
# print("Percentage missing in each column:")
# print(missing_percent_train)

high_missing_cols = missing_percent_train[missing_percent_train > missing_threshold].index.tolist()

print("Columns with more than 40% missing values:")
print(high_missing_cols)

Columns with more than 40% missing values:
['deck']


In [14]:
# to drop the dec column from training and test set, uncomment below 2 lines and execute
X_train = X_train.drop(columns=high_missing_cols)
X_test = X_test.drop(columns=high_missing_cols)

# not dropping just for this video demonstration

**Observation:**
If a column has a large percentage of missing values
for example more than 40 to 60 percent
it often makes sense to drop that column.
Imputation cannot recover meaningful information
when most of the data is missing.

In this dataset we will continue with imputation for teaching,
but in real projects high missing columns are often removed.

**7. Identify numerical and categorical columns**

In [15]:
cols_list = X_train.columns.to_list()
print(cols_list)

['pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'adult_male', 'embark_town', 'alone']


In [16]:
num_cols = X_train.select_dtypes([np.number]).columns.to_list()
cat_cols = [x for x in X_train.columns if x not in num_cols]

print("Numerical columns:", num_cols)
print("Categorical columns:", cat_cols)

Numerical columns: ['pclass', 'age', 'sibsp', 'parch', 'fare']
Categorical columns: ['sex', 'adult_male', 'embark_town', 'alone']


**8. Impute missing values the correct way (TRAIN only)**

In [17]:
X_train_imputed = X_train.copy(deep=True)
X_test_imputed = X_test.copy()

**8.1. Numeric Imputation**

In [18]:
X_train.columns

Index(['pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'adult_male',
       'embark_town', 'alone'],
      dtype='object')

In [19]:
numeric_medians = {}

for col in num_cols:
    median_val = X_train_imputed[col].mean()
    numeric_medians[col] = median_val
    X_train_imputed[col] = X_train_imputed[col].fillna(median_val)
    X_test_imputed[col] = X_test_imputed[col].fillna(median_val)

In [20]:
X_train_imputed.isna().sum()

pclass         0
sex            0
age            0
sibsp          0
parch          0
fare           0
adult_male     0
embark_town    2
alone          0
dtype: int64

**8.2. Categorical Imputation (mode)**

In [21]:
categorical_modes = {}

for col in cat_cols:
    mode_val = X_train_imputed[col].mode().iloc[0]
    categorical_modes[col] = mode_val
    
    print(f"Filling categorical column {col} with TRAIN mode: {mode_val}")
    X_train_imputed[col] = X_train_imputed[col].fillna(mode_val)
    X_test_imputed[col] = X_test_imputed[col].fillna(mode_val)

Filling categorical column sex with TRAIN mode: male
Filling categorical column adult_male with TRAIN mode: True
Filling categorical column embark_town with TRAIN mode: Southampton
Filling categorical column alone with TRAIN mode: True


In [22]:
print("\nMissing values in TRAIN (after):")
print(X_train_imputed.isna().sum())

print("\nMissing values in TEST (after):")
print(X_test_imputed.isna().sum())


Missing values in TRAIN (after):
pclass         0
sex            0
age            0
sibsp          0
parch          0
fare           0
adult_male     0
embark_town    0
alone          0
dtype: int64

Missing values in TEST (after):
pclass         0
sex            0
age            0
sibsp          0
parch          0
fare           0
adult_male     0
embark_town    0
alone          0
dtype: int64


**9. Recombine features and target & save it for future use**

In [23]:
train_imputed_df = X_train_imputed.copy()
test_imputed_df = X_test_imputed.copy()

train_imputed_df[target] = y_train.values
test_imputed_df[target] = y_test.values

train_imputed_df.to_csv("titanic_train_imputed.csv", index=False)
test_imputed_df.to_csv("titanic_test_imputed.csv", index=False)

print("\nSaved imputed data:")
print(" - titanic_train_imputed.csv")
print(" - titanic_test_imputed.csv")


Saved imputed data:
 - titanic_train_imputed.csv
 - titanic_test_imputed.csv
